In [174]:
import pandas as pd

**Preparing edgelist for SNA takes the following steps:** 

For edgelists on bilateral trade data: 
> 1. Loading total, unique, reproducible trade data and the fixed dataset from the cleaned subfolder of the data folder.
> 2. Checking for mismatched countries in all data frames using inner join method as the study will consider only the countries that are present in all data frames
> 3. Making an edge list for total goods trade and saving it in the 'cleaned' subfolder of the folder 'data'
> 4. Making an edge list for unique cultural goods and saving it in the 'cleaned' subfolder of the folder 'data'
> 5. Making an edge list for reproducible cultural goods and saving it in the 'cleaned' subfolder of the folder 'data'
> 6. Saving the edgelists into the 'cleaned' subfolder of 'data' folder

For fixed undireced distance, common language, contiguity, colonizer and common colonizer edgelists: 
> 1. Keeping only the relevant column with the origin and destination column to create a primary data frame
> 2. Checking for a single pair of countries whether the value is same for (A, B) and (B, A)
> 3. Adding log transformed distance column to the distance edgelist
> 4. Saving the edgelists into the 'cleaned' subfolder of 'data' folder

`So this notebook will end up making eleven different edgelists- three for bilateral trade, three for correspondng hysteresis effect and five for bilateral time invariant factors.`

In [178]:
noncultural = pd.read_csv('../data/cleaned/noncultural.csv')
repro = pd.read_csv('../data/cleaned/reproducible.csv')
unique = pd.read_csv('../data/cleaned/unique.csv')
fixed = pd.read_csv('../data/cleaned/fixed.csv')

In [179]:
print(set(noncultural['iso_o'].unique()) - set(unique['iso_o'].unique()))
print(set(noncultural['iso_o'].unique()) - set(repro['iso_o'].unique()))
print(set(repro['iso_o'].unique()) - set(unique['iso_o'].unique()))
print(set(noncultural['iso_d'].unique()) - set(unique['iso_d'].unique()))
print(set(noncultural['iso_d'].unique()) - set(repro['iso_d'].unique()))
print(set(repro['iso_d'].unique()) - set(unique['iso_d'].unique()))

set()
set()
set()
set()
set()
set()


##### To ensure seamless production of square matrices from the generated edgelists, I am checking if the exporter and importer countries are the same for all three datasets (total, unique and reproducible).

In [180]:
assert noncultural['iso_o'].nunique() == noncultural['iso_d'].nunique() # checking if the number of items are same in both sets

In [182]:
assert unique['iso_o'].nunique() == unique['iso_d'].nunique() # checking if the number of items are same in both sets

In [184]:
assert repro['iso_o'].nunique() == repro['iso_d'].nunique()

In [187]:
print(noncultural['iso_o'].nunique())
print(noncultural['iso_d'].nunique())
print(repro['iso_o'].nunique())
print(repro['iso_d'].nunique())
print(unique['iso_o'].nunique())
print(unique['iso_d'].nunique())
print(fixed['iso_o'].nunique())

204
204
204
204
204
204
224


###### So, there are 204 countries in trade datasets, where the fixed dataset has 224 countries.

In [188]:
f_u_ego = set(fixed['iso_o'].unique()) - set(unique['iso_o'].unique())
f_r_ego = set(fixed['iso_o'].unique()) - set(repro['iso_o'].unique())
f_t_ego = set(fixed['iso_o'].unique()) - set(noncultural['iso_o'].unique())
f_u_alter = set(fixed['iso_d'].unique()) - set(unique['iso_d'].unique())
f_r_alter = set(fixed['iso_d'].unique()) - set(repro['iso_d'].unique())
f_t_alter = set(fixed['iso_d'].unique()) - set(noncultural['iso_d'].unique())

In [189]:
len(f_u_ego) == len(f_r_ego) == len(f_t_ego) == len(f_u_alter) == len(f_r_alter) == len(f_t_alter)

True

In [190]:
u_f_ego = set(unique['iso_o'].unique()) - set(fixed['iso_o'].unique())
r_f_ego = set(repro['iso_o'].unique()) - set(fixed['iso_o'].unique())
t_f_ego = set(noncultural['iso_o'].unique()) - set(fixed['iso_o'].unique())
u_f_alter = set(unique['iso_d'].unique()) - set(fixed['iso_d'].unique())
r_f_alter = set(repro['iso_d'].unique()) - set(fixed['iso_d'].unique())
t_f_alter = set(noncultural['iso_d'].unique()) - set(fixed['iso_d'].unique())

In [191]:
len(u_f_ego) == len(r_f_ego) == len(t_f_ego) == len(u_f_alter) == len(r_f_alter) == len(t_f_alter)

True

In [192]:
len(u_f_ego)

12

In [193]:
len(f_u_ego)

32

###### So, the difference from the fixed dataset to trade datasets are all same and reverse is true as well. 

###### So, there are 192 countries present in both datasets. This study will prune other countries for seamless social network analysis. 

In [194]:
# Finding out common elements of both sets:
# Reference: https://www.geeksforgeeks.org/python-print-common-elements-two-lists/
m = list(set(fixed['iso_o'].unique()) & set(unique['iso_o'].unique()))
n = list(set(fixed['iso_o'].unique()) & set(repro['iso_o'].unique()))
o = list(set(fixed['iso_o'].unique()) & set(noncultural['iso_o'].unique()))
p = list(set(fixed['iso_d'].unique()) & set(unique['iso_d'].unique()))
q = list(set(fixed['iso_d'].unique()) & set(repro['iso_d'].unique()))
r = list(set(fixed['iso_d'].unique()) & set(noncultural['iso_d'].unique()))

In [195]:
len(m) == len(n) == len(o) == len(p) == len(q) == len(r)

True

In [196]:
len(m)

192

In [197]:
from collections import Counter

In [198]:
Counter(m) == Counter(n) == Counter(o) == Counter(p) == Counter(q) == Counter(r)

True

##### keeping only the common countries in all dyad level datasets.

In [199]:
unique_com = unique[unique['iso_o'].isin(m) & unique['iso_d'].isin(m)]
repro_com = repro[repro['iso_o'].isin(m) & repro['iso_d'].isin(m)]
total_com = noncultural[noncultural['iso_o'].isin(m) & noncultural['iso_d'].isin(m)]
fixed_com = fixed[fixed['iso_o'].isin(m) & fixed['iso_d'].isin(m)]

In [200]:
print(total_com['iso_o'].nunique())
print(total_com['iso_d'].nunique())
print(repro_com['iso_o'].nunique())
print(repro_com['iso_d'].nunique())
print(unique_com['iso_o'].nunique())
print(unique_com['iso_d'].nunique())
print(fixed_com['iso_o'].nunique())
print(fixed_com['iso_d'].nunique())

192
192
192
192
192
192
192
192


### Edgelist for non-cultural trade & Hysteresis effect

In [201]:
set(total_com['iso_o'].unique()) == set(total_com['iso_d'].unique())

True

In [203]:
total_com.head(2)

,iso_o,iso_d,TIME_PERIOD,noncultural_trade,product_count,hysteresis_noncultural_trade
0,ABW,AFG,2017,0.069146,10,0.000000
1,ABW,AFG,2018,0.422331,15,0.069146


In [204]:
edgelist_total = total_com.pivot_table(index = ['iso_o', 'iso_d'], columns = 'TIME_PERIOD', values = 'noncultural_trade', fill_value = 0).reset_index()

In [205]:
edgelist_total_rounded = edgelist_total.round(4)

In [206]:
assert total_com.groupby(['iso_o', 'iso_d']).ngroups == edgelist_total_rounded.shape[0]

In [207]:
edgelist_total_rounded.to_csv("../data/cleaned/total_edgelist.csv", encoding='utf-8', index=False)

In [208]:
total_com.head(3)

,iso_o,iso_d,TIME_PERIOD,noncultural_trade,product_count,hysteresis_noncultural_trade
0,ABW,AFG,2017,0.069146,10,0.000000
1,ABW,AFG,2018,0.422331,15,0.069146
2,ABW,AFG,2019,0.278743,29,0.443075


In [209]:
edgelist_total_hyst = total_com.pivot_table(index = ['iso_o', 'iso_d'], columns = 'TIME_PERIOD', values = 'hysteresis_noncultural_trade', fill_value = 0).reset_index()

In [210]:
assert total_com.groupby(['iso_o', 'iso_d']).ngroups == edgelist_total_rounded.shape[0] == edgelist_total_hyst.shape[0]

In [211]:
edgelist_total_hyst.head(3)

TIME_PERIOD,iso_o,iso_d,1995,1996,1997,1998,1999,2000,2001,2002,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,ABW,AFG,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.069146,0.443075,0.0,0.0,0.0,0.000000,0.000000
1,ABW,AGO,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.465854,0.0,0.0,0.0,0.145892,0.057916
2,ABW,ALB,0.0,0.002297,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.015659,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000


In [212]:
edgelist_total_hyst.to_csv("../data/cleaned/edgelist_total_hyst.csv", encoding='utf-8', index=False)

### Edgelist for unique cultural goods trade & Hysteresis Effect

In [213]:
set(unique_com['iso_o'].unique()) == set(unique_com['iso_d'].unique())

True

In [214]:
edgelist_unique = unique_com.pivot_table(index = ['iso_o', 'iso_d'], columns = 'TIME_PERIOD', values = 'unique', fill_value = 0).reset_index()

In [215]:
edgelist_unique_rounded = edgelist_unique.round(4)

In [216]:
edgelist_unique_rounded['iso_o'].nunique()

192

In [217]:
assert unique_com.groupby(['iso_o', 'iso_d']).ngroups == edgelist_unique_rounded.shape[0] # making sure we have same number of dyads in both panel data and edgelist

In [218]:
edgelist_unique_rounded.to_csv('../data/cleaned/unique_edgelist.csv', encoding = 'utf-8', index = False) 

In [219]:
unique_com.head(3)

,iso_o,iso_d,TIME_PERIOD,unique,product_count,hysteresis_unique
0,ABW,ARE,2023,0.000113,1,0.000000
1,ABW,AUT,2013,0.002537,2,0.000000
2,ABW,AUT,2014,0.001343,1,0.002537


In [220]:
edgelist_unique_hyst = unique_com.pivot_table(index = ['iso_o', 'iso_d'], columns = 'TIME_PERIOD', values = 'hysteresis_unique', fill_value = 0).reset_index()

In [221]:
edgelist_unique_hyst.head(3)

TIME_PERIOD,iso_o,iso_d,1995,1996,1997,1998,1999,2000,2001,2002,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,ABW,ARE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
1,ABW,AUT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2,ABW,BEL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002836,0.0


In [222]:
assert unique_com.groupby(['iso_o', 'iso_d']).ngroups == edgelist_unique_rounded.shape[0] == edgelist_unique_hyst.shape[0]

In [223]:
edgelist_unique_hyst.to_csv("../data/cleaned/edgelist_unique_hyst.csv", encoding='utf-8', index=False)

### Edgelist for reproducible cultural goods trade & Hysteresis effect

In [224]:
set(repro_com['iso_o'].unique()) == set(repro_com['iso_d'].unique())

True

In [225]:
edgelist_repro = repro_com.pivot_table(index = ['iso_o', 'iso_d'], columns = 'TIME_PERIOD', values = 'reproducible', fill_value = 0).reset_index()

In [226]:
edgelist_repro_rounded = edgelist_repro.round(4)

In [227]:
edgelist_repro_rounded['iso_o'].nunique()

192

In [228]:
assert repro_com.groupby(['iso_o', 'iso_d']).ngroups == edgelist_repro_rounded.shape[0]

In [229]:
edgelist_repro_rounded.to_csv('../data/cleaned/reproducible_edgelist.csv', encoding = 'utf-8', index = False) 

In [230]:
repro_com.head(2)

,iso_o,iso_d,TIME_PERIOD,reproducible,product_count,hysteresis_reproducible
0,ABW,AFG,2019,0.001492,1,0.0
1,ABW,ARG,2000,0.000015,1,0.0


In [231]:
edgelist_repro_hyst = repro_com.pivot_table(index = ['iso_o', 'iso_d'], columns = 'TIME_PERIOD', values = 'hysteresis_reproducible', fill_value = 0).reset_index()

In [232]:
assert repro_com.groupby(['iso_o', 'iso_d']).ngroups == edgelist_repro_rounded.shape[0] == edgelist_repro_hyst.shape[0]

In [233]:
repro_com.head(2)

,iso_o,iso_d,TIME_PERIOD,reproducible,product_count,hysteresis_reproducible
0,ABW,AFG,2019,0.001492,1,0.0
1,ABW,ARG,2000,0.000015,1,0.0


In [234]:
edgelist_repro_hyst.head(3)

TIME_PERIOD,iso_o,iso_d,1995,1996,1997,1998,1999,2000,2001,2002,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,ABW,AFG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ABW,ARG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000015,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ABW,ATG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [235]:
edgelist_repro_hyst.to_csv("../data/cleaned/edgelist_repro_hyst.csv", encoding='utf-8', index=False)

### Edgelist for contiguity

In [236]:
set(fixed_com['iso_o'].unique()) == set(fixed_com['iso_d'].unique())

True

In [237]:
fixed_com_con = fixed_com[['iso_o', 'iso_d', 'contig']]

In [238]:
fixed_com_con.shape

(36672, 3)

In [239]:
print(fixed_com_con[(fixed_com_con['iso_o'] == 'BGD') & (fixed_com_con['iso_d'] == 'IND')])
print(fixed_com_con[(fixed_com_con['iso_o'] == 'IND') & (fixed_com_con['iso_d'] == 'BGD')])

     iso_o iso_d  contig
4105   BGD   IND       1
      iso_o iso_d  contig
20534   IND   BGD       1


In [240]:
print(fixed_com_con['iso_o'].nunique())
print(fixed_com_con['iso_o'].nunique())
print(fixed_com_con.groupby(['iso_o', 'iso_d']).ngroups)

192
192
36672


In [241]:
fixed_com_con.to_csv('../data/cleaned/contiguity_edgelist.csv', encoding = 'utf-8', index = False) 

### Edgelist for distance

In [242]:
fixed_com.head(2)

,iso_o,iso_d,contig,comlang_off,colony,comcol,dist,log_dist,country,landlocked,continent,lat,lon,langoff_1,colonizer1,category,development
0,ABW,AFG,0,0,0,0,13257.810,9.492342,Aruba,0,America,12.55,-70.099998,Dutch,NLD,Emerging and Developing Economies,0.0
1,ABW,AGO,0,0,0,0,9516.913,9.160826,Aruba,0,America,12.55,-70.099998,Dutch,NLD,Emerging and Developing Economies,0.0


In [243]:
distance = fixed_com[['iso_o', 'iso_d', 'log_dist']]

In [244]:
print(distance[(distance['iso_o'] == 'BGD') & (distance['iso_d'] == 'IND')])
print(distance[(distance['iso_o'] == 'IND') & (distance['iso_d'] == 'BGD')])

     iso_o iso_d  log_dist
4105   BGD   IND  7.259776
      iso_o iso_d  log_dist
20534   IND   BGD  7.259776


In [245]:
distance.shape

(36672, 3)

In [246]:
print(distance['iso_o'].nunique())
print(distance['iso_d'].nunique())

192
192


In [247]:
distance.head(2)

,iso_o,iso_d,log_dist
0,ABW,AFG,9.492342
1,ABW,AGO,9.160826


In [248]:
distance.to_csv('../data/cleaned/distance_edgelist.csv', encoding = 'utf-8', index = False) 

### Edgelist for common colonizer

In [249]:
comcol = fixed_com[['iso_o', 'iso_d', 'comcol']]

In [250]:
comcol.shape

(36672, 3)

In [251]:
print(comcol[(comcol['iso_o'] == 'BGD') & (comcol['iso_d'] == 'IND')])
print(comcol[(comcol['iso_o'] == 'IND') & (comcol['iso_d'] == 'BGD')])

     iso_o iso_d  comcol
4105   BGD   IND       1
      iso_o iso_d  comcol
20534   IND   BGD       1


In [252]:
print(comcol['iso_o'].nunique())
print(comcol['iso_d'].nunique())

192
192


In [253]:
comcol.to_csv('../data/cleaned/comcol_edgelist.csv', encoding = 'utf-8', index = False)

### Edgelist for colonizer

In [254]:
col = fixed_com[['iso_o', 'iso_d', 'colony']]

In [255]:
col.shape

(36672, 3)

In [256]:
print(col[(col['iso_o'] == 'BGD') & (col['iso_d'] == 'GBR')])
print(col[(col['iso_o'] == 'IND') & (col['iso_d'] == 'GBR')])

     iso_o iso_d  colony
4084   BGD   GBR       1
      iso_o iso_d  colony
20587   IND   GBR       1


In [257]:
print(col['iso_o'].nunique())
print(col['iso_d'].nunique())

192
192


In [258]:
col.to_csv('../data/cleaned/colonizer_edgelist.csv', encoding = 'utf-8', index = False)

### Edgelist for common language

In [259]:
lang = fixed_com[['iso_o', 'iso_d', 'comlang_off']]

In [260]:
lang.shape

(36672, 3)

In [261]:
print(lang[(lang['iso_o'] == 'BGD') & (lang['iso_d'] == 'IND')])
print(lang[(lang['iso_o'] == 'IND') & (lang['iso_d'] == 'BGD')])

     iso_o iso_d  comlang_off
4105   BGD   IND            0
      iso_o iso_d  comlang_off
20534   IND   BGD            0


In [262]:
print(lang['iso_o'].nunique())
print(lang['iso_d'].nunique())

192
192


In [263]:
lang.to_csv('../data/cleaned/language_edgelist.csv', encoding = 'utf-8', index = False)